# GPU scaling benchmark: real estimation & optimization pipelines

Where is the CPU/GPU break-even for the *real* Twin4Build pipelines?  This
notebook sweeps model size N (thermal zones in a chain, consecutive zones
coupled by a wall; compile-time fusion turns the whole chain into ONE
state-space block of `3N-1` states) and times, at every N, on
cpu/fp64, cuda/fp64 and cuda/fp32:

- **Estimation** -- `Estimator.estimate` (scipy SLSQP + AD, fast
  single-shooting objective): one `C_air` per zone and one wall `C` per wall
  calibrated against synthetic noisy zone-temperature measurements
  (144 timesteps).  Metric: **seconds per objective+gradient evaluation**.
- **Optimization** -- `Optimizer.optimize` (scipy SLSQP + AD, fast composed
  objective): every zone's heater schedule (24 hourly values each) chosen to
  minimize energy under a comfort constraint.  Metric: **seconds per SLSQP
  iteration**.

**Setup**: Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.
The full sweep takes roughly 15-30 minutes.

**Troubleshooting**: if an import fails with a numpy error after the install,
Runtime > Restart session and Run all again.  To pick up new branch commits,
Runtime > Disconnect and delete runtime first.


In [ ]:
# Clone (or update) the PR branch and install without touching Colab's
# preinstalled scientific stack (replacing numpy mid-session breaks the kernel).
import importlib.metadata as _md
_pins = " ".join(
    f'"{_p}=={_md.version(_p)}"'
    for _p in ("numpy", "scipy", "pandas", "matplotlib")
)
!git clone --depth 1 -b feature/gpu-device-support https://github.com/JBjoernskov/Twin4Build.git 2>/dev/null || git -C Twin4Build pull
!pip install -q ./Twin4Build {_pins}

import sys
sys.path.insert(0, "Twin4Build")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- the sweep will run CPU-only (no break-even measurable).")

CONFIGS = [("cpu", torch.float64)]
if torch.cuda.is_available():
    CONFIGS += [("cuda", torch.float64), ("cuda", torch.float32)]
EST_SIZES = [1, 2, 4, 8, 16, 32, 64]
OPT_SIZES = [1, 2, 4, 8, 16, 32]


In [ ]:
from twin4build.examples.gpu_benchmark_scaling import (
    breakeven,
    run_estimation_case,
    run_optimization_case,
    sweep,
)

df_est = sweep(run_estimation_case, EST_SIZES, CONFIGS, maxiter=2)
df_est


In [ ]:
df_opt = sweep(run_optimization_case, OPT_SIZES, CONFIGS, maxiter=5)
df_opt


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (df, metric, title) in zip(
    axes,
    [
        (df_est, "s_per_eval", "Estimation: s per objective+gradient eval"),
        (df_opt, "s_per_iter", "Optimization: s per SLSQP iteration"),
    ],
):
    for (device, dtype), grp in df.groupby(["device", "dtype"]):
        grp = grp.sort_values("n_zones")
        ax.plot(
            grp["n_zones"], grp[metric], marker="o",
            label=f"{device}/{dtype}",
        )
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("zones N  (fused model: 3N-1 states)")
    ax.set_ylabel("wall time [s]")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

for df, metric, name in [
    (df_est, "s_per_eval", "estimation"),
    (df_opt, "s_per_iter", "optimization"),
]:
    for cfg in [("cuda", "float64"), ("cuda", "float32")]:
        n = breakeven(df, metric, gpu_config=cfg)
        where = f"N >= {n}" if n is not None else "not reached in this sweep"
        print(f"{name:>12} | {cfg[0]}/{cfg[1]:<8} beats cpu/float64 at: {where}")


## How to read the results

- Both pipelines are dominated by the sequential rollout: 144 (estimation)
  or 24 (optimization) dependent one-step maps, each a `matrix_exp` +
  mat-vec on the fused `(3N-1)`-state block, plus one reverse-mode sweep for
  the gradient.
- **Small N**: per-op work is microseconds; the GPU pays a kernel-launch
  latency (~5-10 us) on every op and loses.  This region belongs to the CPU.
- **Growing N**: the `O(n^3)` `matrix_exp` starts to dominate and the GPU's
  arithmetic advantage compounds -- the cuda curves flatten while the cpu
  curve rises.  The printed break-even is where they cross.
- **fp32 vs fp64**: on consumer GPUs (T4: fp64 at 1/32 rate) the fp32 curve
  should cross substantially earlier.  Check the estimation `fast` column
  stayed `True`: it confirms the fast objective still passed its build on
  every configuration.
- **`gpu_util_pct`** is the fraction of wall-clock time the GPU was actually
  executing a kernel (NVML `utilization.gpu`, sampled at 10 Hz during the
  run; NaN for cpu rows).  At small N expect single-digit percentages --
  the run is dominated by Python overhead and kernel-launch latency, the GPU
  mostly idles.  The break-even N is roughly where this number gets large:
  once the GPU is busy most of the wall time, adding states is nearly free
  for the cuda curves while the cpu curve keeps rising.
- **`gpu_util_pct`** is the fraction of wall-clock time the GPU was actually
  executing a kernel (NVML `utilization.gpu`, sampled at 10 Hz during the
  run; NaN for cpu rows).  At small N expect single-digit percentages --
  the run is dominated by Python overhead and kernel-launch latency, the GPU
  mostly idles.  The break-even N is roughly where this number gets large:
  once the GPU is busy most of the wall time, adding states is nearly free
  for the cuda curves while the cpu curve keeps rising.
- **`gpu_util_pct`** is the fraction of wall-clock time the GPU was actually
  executing a kernel (NVML `utilization.gpu`, sampled at 10 Hz during the
  run; NaN for cpu rows).  At small N expect single-digit percentages --
  the run is dominated by Python overhead and kernel-launch latency, the GPU
  mostly idles.  The break-even N is roughly where this number gets large:
  once the GPU is busy most of the wall time, adding states is nearly free
  for the cuda curves while the cpu curve keeps rising.
- Not covered here: *batched* workloads (multi-start, scenarios, portfolios
  via `n_c`), where the GPU wins at much smaller per-model sizes -- see
  `gpu_benchmark_estimation.ipynb` / `gpu_benchmark_optimizer.ipynb` Part B.
